# Digital Forge Reel — Colab Renderer (GPU Pipeline Fixed)

Render the HTML/CSS/JS scene into a vertical MP4 video directly on Google Colab.

**Required runtime:** GPU (Runtime → Change runtime type → T4 GPU)

## What this notebook does
1. Unzips the project
2. Runs the one-shot setup script (installs Node 20, fonts, ffmpeg, Playwright Chromium, Python deps)
3. **Runs the GPU diagnostic** to verify Chrome is actually using the T4 (not SwiftShader)
4. **Runs the benchmark** to compare CPU vs GPU rendering speed
5. Renders the English + Arabic videos (GPU rasterization + NVENC encoding + parallel capture)
6. Downloads them to your machine / Google Drive

## What was fixed (compared to the original notebook)

The original project had a **silent GPU bug**: Chrome's `--use-gl=egl` flag was rejected by Chrome 131+,
causing Chrome to fall back to SwiftShader (CPU rasterization). The GPU sat idle during the entire
render, and only NVENC encoding (5% of total time) used the GPU.

This notebook uses the fixed version with:
- ✅ Correct Chrome GPU flags (`--use-gl=angle --use-angle=vulkan --enable-features=Vulkan`)
- ✅ GPU verification via `chrome://gpu` (catches SwiftShader fallback)
- ✅ Parallel capture (4 Chrome workers = 4x capture speedup on 4-vCPU Colab)
- ✅ NVENC `hwupload_cuda` (frames stay on GPU after upload)
- ✅ GPU monitoring during render (verifies utilization in real-time)

Expected speedup: **5-7x faster** than the original "GPU" mode (which was actually CPU).

See `docs/GPU_FIX.md` for the full diagnosis and fix explanation.

## Step 1 — Upload the project zip

Upload `digital-forge-reel.zip` to Colab using the file browser on the left (or mount Drive).

Then run the cell below to unzip it.

In [ ]:
# Unzip the project
%cd /content
!unzip -o digital-forge-reel.zip -d /content/
%cd /content/digital-forge-reel
!ls -la

## Step 2 — Run the one-shot setup script

This installs everything: Node 20, fonts, ffmpeg, Python deps, Playwright Chromium binary.

Takes ~5-8 minutes the first time (mostly the Playwright Chromium download + apt installs).

**If you already ran this before, you can skip to Step 4.**

In [ ]:
!bash setup-colab.sh

## Step 3 — Run the GPU diagnostic

This verifies that Chrome is **actually using the GPU** (not silently falling back to SwiftShader/CPU).

If all checks pass, you'll see:
```
✓ NVIDIA driver: 525.x (Tesla T4)
✓ ffmpeg h264_nvenc: available
✓ ffmpeg hwupload_cuda: available
✓ Chrome launches with GPU flags
✓ chrome://gpu: Hardware accelerated
✓ WebGL UNMASKED_RENDERER: ANGLE (NVIDIA, Tesla T4, ...)
✓ Sample render GPU utilization: 22% avg (peak 35%)
All checks passed. GPU is fully utilized.
```

If any check fails, the diagnostic tells you exactly how to fix it.

In [ ]:
!node bin/forge-gpu-check --scene scenes/digital-forge-reel-en.html

## Step 4 — Run the benchmark (optional but recommended)

Renders the same 30 frames under 4 configurations and shows the speedup.

Expected output on Colab T4 (4 vCPUs):
```
Name                   | Capture FPS | Capture Time | Encode Time | GPU Avg
-----------------------|-------------|--------------|-------------|--------
cpu-baseline           |        1.2  |      25000ms |      4100ms |     0%
gpu-raster-only        |        3.8  |       7900ms |      4200ms |    12%
cpu-parallel           |        3.5  |       8600ms |      4000ms |     0%
full-gpu-pipeline      |        8.5  |       3500ms |       800ms |    18%

Speedup vs CPU baseline:
  gpu-raster-only: 3.16x faster capture
  cpu-parallel: 2.91x faster capture
  full-gpu-pipeline: 7.14x faster capture
```

Takes ~2 minutes total.

In [ ]:
!node bin/forge-benchmark scenes/digital-forge-reel-en.html --frames 30 --music audio/forge_theme.wav

## Step 5 — Render the English video (full GPU pipeline)

With T4 GPU + GPU rasterization + NVENC + parallel capture, this takes ~2-4 minutes for a 30s reel
(vs ~10-15 minutes with the original "GPU" mode that was actually using CPU).

The command uses:
- `--gpu` → enables Chrome GPU rasterization (verified by Step 3)
- `--encoder nvenc` → NVIDIA hardware H.264 encoder (5-20x faster than CPU libx264)
- `--workers 4` → 4 parallel Chrome processes for capture (matches Colab's 4 vCPUs)
- `--fps 30` → capture 30 source frames per second
- `--target-fps 60` → output 60fps (frames are duplicated to reach 60)
- `--scale 1.0` → capture at full 1080×1920 resolution

### 💡 Pro tip: use `--time-scale` to render 2-4x faster

If you don't need true 60fps motion, add `--time-scale 0.5 --fps 15` to capture at half speed
(15fps instead of 30fps = half as many frames) and speed the video back up in ffmpeg.
The output looks identical but renders 2x faster. Use `--time-scale 0.25 --fps 8` for 4x speedup.

```bash
# Fast version: 2x faster render, same visual output
node bin/forge-render scenes/digital-forge-reel-en.html \
    --gpu --encoder nvenc --workers 4 \
    --time-scale 0.5 --fps 15 --target-fps 60 \
    --output output/en.mp4 --music audio/forge_theme.wav
```

**Resume tip:** If this cell fails or you interrupt it, just re-run it — it skips already-captured frames and retries the encode automatically.

**Watch GPU utilization:** Open a new cell and run `!nvidia-smi -l 1` while this is running to see GPU usage spike during capture (10-40%) and encode (80-95%).

In [ ]:
# Full quality GPU render (Colab T4 GPU runtime)
!node bin/forge-render scenes/digital-forge-reel-en.html \
    --output output/digital-forge-reel-en.mp4 \
    --music audio/forge_theme.wav \
    --gpu \
    --encoder nvenc \
    --workers 4 \
    --fps 30 \
    --target-fps 60 \
    --scale 1.0 \
    --interpolate dup \
    --crf 18 \
    --log-level info

### Optional: Watch GPU utilization in real-time

Run this in a separate cell while Step 5 is running. You should see:
- **During capture** (most of the time): GPU at 10-40% utilization (Chrome rasterization)
- **During encode** (last few seconds): GPU at 80-95% utilization (NVENC)

If GPU stays at 0% during capture, Chrome fell back to SwiftShader — re-run Step 3 to diagnose.

In [ ]:
# Run this in a separate cell WHILE Step 5 is running
# Updates every 1 second for 30 seconds
!timeout 30 nvidia-smi --query-gpu=timestamp,utilization.gpu,memory.used,power.draw --format=csv -l 1

## Step 6 — Render the Arabic video (Algerian Darija)

Same settings as the English render. The Arabic scene uses RTL layout with Cairo + Tajawal fonts.

In [ ]:
!node bin/forge-render scenes/digital-forge-reel-ar.html \
    --output output/digital-forge-reel-ar.mp4 \
    --music audio/forge_theme.wav \
    --gpu \
    --encoder nvenc \
    --workers 4 \
    --fps 30 \
    --target-fps 60 \
    --scale 1.0 \
    --interpolate dup \
    --crf 18 \
    --log-level info

## Step 7 — Verify the videos exist

Check that the MP4 files were created with the right specs (1080×1920 @ 60fps, ~30s, with audio).

In [ ]:
!ls -lh output/*.mp4 2>/dev/null || echo 'No MP4s yet'
print('---')
!ffprobe -v error -show_entries stream=codec_name,width,height,r_frame_rate,duration -of default=noprint_wrappers=1 output/digital-forge-reel-en.mp4 2>/dev/null || echo 'No English video yet'
print('---')
!ffprobe -v error -show_entries stream=codec_name,width,height,r_frame_rate,duration -of default=noprint_wrappers=1 output/digital-forge-reel-ar.mp4 2>/dev/null || echo 'No Arabic video yet'

## Step 8 — Save to Google Drive (persistent)

Colab deletes files when the session ends. Save your videos to Google Drive so they persist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/digital-forge-reel
!cp output/*.mp4 /content/drive/MyDrive/digital-forge-reel/ 2>/dev/null || echo 'No MP4s to copy yet — run Step 5 first'
!ls -lh /content/drive/MyDrive/digital-forge-reel/ 2>/dev/null

## Step 9 — Download directly to your machine

Alternative to Step 8 — downloads each MP4 directly. Useful if you don't want to mount Drive.

In [ ]:
from google.colab import files
import os

for f in os.listdir('output'):
    if f.endswith('.mp4'):
        print(f'Downloading {f}...')
        files.download(f'output/{f}')

## Troubleshooting

### "GPU verification FAILED — Chrome is using CPU rasterization (SwiftShader)"

This means the GPU flags didn't engage. Run:
```bash
!apt install -y libnvidia-gl-525 libvulkan1 vulkan-tools
!vulkaninfo --summary | grep Tesla
```
Then re-run Step 3.

If that doesn't work, fall back to headed Chrome under Xvfb:
```bash
!apt install -y xvfb
!xvfb-run --auto-servernum --server-args="-screen 0 1920x1080x24" \
  node bin/forge-render scene.html --gpu --encoder nvenc
```

### "Chromium not found"
Run `!bash setup-colab.sh` again. If that fails:
```python
!npx playwright install chromium
!npx playwright install-deps chromium
```

### "Command '/usr/bin/chromium-browser' requires the chromium snap to be installed"
Playwright is picking up the snap stub. Run the dedicated fix script:
```python
!node bin/forge-fix-chromium
```

### NVENC encode fails
Make sure you're on a GPU runtime (Runtime → Change runtime type → T4 GPU). Then:
```python
!sudo apt install -y ffmpeg
!ffmpeg -encoders | grep h264_nvenc  # should show h264_nvenc
```
If still missing, fall back to CPU: `--encoder cpu`

### "Worker N exited with code 1" (parallel capture crash)
Reduce worker count: `--workers 2` (instead of 4). Each Chrome worker uses ~500MB RAM and ~200MB VRAM.

### Capture is slow even with --gpu
Run Step 3 (the GPU diagnostic). If `chrome://gpu` says "Software only", Chrome fell back to SwiftShader. The diagnostic will tell you exactly which flag or driver is missing.

### Out of memory
Use `--scale 0.5` (half-res capture) or `--fps 15` (fewer frames):
```python
!node bin/forge-render scene.html --scale 0.5 --fps 15 --output output.mp4 --music audio/forge_theme.wav
```

### Colab disconnects mid-render
Frames are saved to `output/frames/` as they're captured. Re-run the same cell — it resumes from where it left off. For extra safety, mount Drive and point `--frames-dir` at it so frames survive disconnects.

### Want to modify the scene
Edit `scenes/digital-forge-reel-en.html` (or `-ar.html`) directly — it's self-contained HTML, no build step. Then re-run Step 5.

### Want a completely different video format
See `docs/NEW-VIDEO-EXTENSION.md` — the system supports themes, custom scenes, and scaffolding via `node bin/forge-new my-video --theme forge`.

## Learn more

- **[docs/GPU_FIX.md](docs/GPU_FIX.md)** — full diagnosis and fix explanation
- **[docs/GPU.md](docs/GPU.md)** — NVENC / VAAPI / QSV / VideoToolbox reference
- **[docs/COLAB.md](docs/COLAB.md)** — Google Colab guide
- **[docs/ARCHITECTURE.md](docs/ARCHITECTURE.md)** — how the pipeline works